# ARC-Hydra completed-runtime input probe

Private CPU-only diagnostic of the exact completed build output mount and checksums. No model, competition attachment, GPU or network required. Runtime: Pennyroyal/SGLang contributors; original agent infrastructure attribution: Daniel Franzen, Apache-2.0.

In [ ]:
from pathlib import Path
import json, hashlib, datetime
namespace={"__name__":"runtime_input_probe"}
exec('"""Install a CPU-verified overlay into a stopped, isolated model environment.\n\nThis does not test GPU inference and never installs from an online index.\n"""\nimport argparse\nfrom hashlib import sha256\nimport json\nfrom pathlib import Path, PurePosixPath\nimport os\nimport platform\nimport shlex\nimport socket\nimport subprocess\nimport sys\nimport tarfile\n\nPREFIX=Path(\'/tmp/hydra-serving-v3\')\nEXPECTED_COMMIT=\'560edbdca18aedae13f71ae20b3984efa4d22633\'\n\n\ndef digest(path):\n    result=sha256()\n    with path.open(\'rb\') as stream:\n        for block in iter(lambda:stream.read(8*1024*1024),b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef manifest_path(root, relative):\n    value=PurePosixPath(relative)\n    if value.is_absolute() or \'..\' in value.parts or \'\\\\\' in relative:\n        raise ValueError(f\'Unsafe manifest path: {relative}\')\n    path=(root/relative).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError(f\'Manifest path escapes overlay: {relative}\')\n    return path\n\n\ndef verify_bundle(root):\n    if (root/\'BUILD_COMPLETE\').read_text().strip()!=\'CPU_BUILD_AND_OFFLINE_INSTALL_VERIFIED; GPU_INFERENCE_NOT_TESTED\':\n        raise ValueError(\'Missing CPU build/offline-install completion evidence\')\n    provenance=json.loads((root/\'provenance.json\').read_text())\n    if provenance[\'source_commit\']!=EXPECTED_COMMIT:\n        raise ValueError(\'Unexpected runtime source commit\')\n    manifest=json.loads((root/\'manifest.json\').read_text())[\'files\']\n    for relative, expected in manifest.items():\n        path=manifest_path(root,relative)\n        if path.stat().st_size!=expected[\'bytes\'] or digest(path)!=expected[\'sha256\']:\n            raise ValueError(f\'Overlay checksum mismatch: {relative}\')\n    required={\'gcc-toolchain.tar.gz\',\'compiler.lock\',\'requirements-overlay.lock\',\'cuda_overlay.py\'}\n    if not required <= manifest.keys():\n        raise ValueError(\'Overlay manifest is incomplete\')\n    return digest(root/\'manifest.json\')\n\n\ndef run(command, env):\n    print(\'$\',shlex.join(map(str,command)),flush=True)\n    subprocess.run(list(map(str,command)),env=env,check=True)\n\n\ndef install(root, base, venv, uv, port):\n    if sys.platform!=\'linux\' or platform.machine()!=\'x86_64\' or sys.version_info[:2]!=(3,12):\n        raise RuntimeError(\'Overlay requires Linux x86_64 with Python 3.12\')\n    with socket.socket() as probe:\n        if probe.connect_ex((\'127.0.0.1\',port))==0:\n            raise RuntimeError(\'Model server is already listening; refusing an in-place update\')\n    manifest_hash=verify_bundle(root)\n    env=dict(os.environ,UV_OFFLINE=\'1\',UV_PYTHON_DOWNLOADS=\'never\',PYTHONNOUSERSITE=\'1\',\n             HF_HUB_OFFLINE=\'1\',TRANSFORMERS_OFFLINE=\'1\')\n    env.pop(\'PYTHONPATH\',None)\n    compiler=PREFIX/\'compiler\'\n    gcc=PREFIX/\'gcc\'\n    cuda=PREFIX/\'cuda\'\n    marker=PREFIX/\'overlay-installed.json\'\n    identity=dict(manifest_sha256=manifest_hash,venv=str(venv.resolve()))\n    if marker.exists():\n        if json.loads(marker.read_text())!=identity:\n            raise RuntimeError(\'Another overlay is already installed in this prefix\')\n    else:\n        if any(p.exists() for p in (compiler,gcc,cuda)):\n            raise RuntimeError(\'Partial toolchain prefix exists; use a fresh notebook runtime\')\n        PREFIX.mkdir(parents=True,exist_ok=True)\n        with tarfile.open(root/\'gcc-toolchain.tar.gz\',\'r:gz\') as archive:\n            archive.extractall(PREFIX,filter=\'data\')\n        if not (gcc/\'bin/x86_64-conda-linux-gnu-g++\').is_file():\n            raise RuntimeError(\'Compiler archive did not contain the expected prefix\')\n        run([uv,\'venv\',\'--python\',sys.executable,compiler],env)\n        run([uv,\'pip\',\'install\',\'--python\',compiler/\'bin/python\',\'--no-index\',\n             \'--find-links\',root/\'compiler-wheels\',\'-r\',root/\'compiler.lock\'],env)\n        run([compiler/\'bin/python\',root/\'cuda_overlay.py\',cuda],env)\n        run([uv,\'pip\',\'install\',\'--python\',venv/\'bin/python\',\'--no-index\',\n             \'--find-links\',base/\'wheels\',\'--find-links\',root/\'wheels\',\n             \'-r\',root/\'requirements-overlay.lock\'],env)\n    env.update(CC=str(gcc/\'bin/x86_64-conda-linux-gnu-gcc\'),\n               CXX=str(gcc/\'bin/x86_64-conda-linux-gnu-g++\'),\n               CUDAHOSTCXX=str(gcc/\'bin/x86_64-conda-linux-gnu-g++\'),\n               CUDA_HOME=str(cuda),CUDA_PATH=str(cuda),CUDACXX=str(cuda/\'bin/nvcc\'),\n               PATH=f\'{gcc}/bin:{cuda}/bin:{venv}/bin:\'+env.get(\'PATH\',\'\'),\n               LD_LIBRARY_PATH=f\'{gcc}/lib:{cuda}/lib64:\'+env.get(\'LD_LIBRARY_PATH\',\'\'))\n    run([venv/\'bin/python\',\'-m\',\'pip\',\'check\'],env)\n    run([venv/\'bin/python\',root/\'cuda_overlay.py\',cuda,\'--libraries-only\'],env)\n    run([cuda/\'bin/nvcc\',\'--version\'],env)\n    run([venv/\'bin/python\',\'-c\',\n         \'import importlib.metadata as m; \'\n         \'assert m.version("sglang")=="0.5.19+g560edbdca18a"; \'\n         \'assert m.version("flashinfer-python")=="0.7.0.post1"; \'\n         \'from pathlib import Path; import flashinfer; \'\n         \'assert (Path(flashinfer.__file__).parent/"data/aot/fused_moe_120/fused_moe_120.so").is_file(); \'\n         \'print("Runtime overlay installed; GPU inference still unverified")\'],env)\n    marker.write_text(json.dumps(identity,indent=2)+\'\\n\')\n    keys=(\'CC\',\'CXX\',\'CUDAHOSTCXX\',\'CUDA_HOME\',\'CUDA_PATH\',\'CUDACXX\',\'PATH\',\'LD_LIBRARY_PATH\')\n    return {key:env[key] for key in keys}\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser()\n    for flag in (\'overlay\',\'base\',\'venv\',\'uv\',\'environment-output\'):\n        parser.add_argument(\'--\'+flag,type=Path,required=True)\n    parser.add_argument(\'--port\',type=int,default=8001)\n    args=parser.parse_args()\n    environment=install(args.overlay,args.base,args.venv,args.uv,args.port)\n    args.environment_output.write_text(json.dumps(environment,indent=2)+\'\\n\')\n',namespace)
options=[Path('/kaggle/input/notebooks/kunalsinghh25/arc-hydra-pennyroyal-v3-dependency-repair/pennyroyal-v3-overlay'),Path('/kaggle/input/arc-hydra-pennyroyal-v3-dependency-repair/pennyroyal-v3-overlay')]
print('Checking exact completed kernel input mount',flush=True)
print('Input top directories:',sorted(str(p) for p in Path('/kaggle/input').iterdir()),flush=True)
selected=next((p for p in options if (p/'manifest.json').is_file()),None)
if selected is None:
    raise RuntimeError('Completed CPU kernel output did not mount at either supported path')
actual=hashlib.sha256((selected/'manifest.json').read_bytes()).hexdigest()
assert actual=='f1b8e87b99402e8504491bb2d2d107df81b119295d4b919944986eacb2fef7fb', "Unexpected input manifest"
print('Mounted:',selected,'manifest:',actual,flush=True)
verified=namespace['verify_bundle'](selected)
assert verified==actual
files=json.loads((selected/'manifest.json').read_text())['files']
result=dict(kernel_input='kunalsinghh25/arc-hydra-pennyroyal-v3-dependency-repair/6',mounted_path=str(selected),manifest_sha256=actual,verified_files=len(files),verified_bytes=sum(f['bytes'] for f in files.values()),checked_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),boundary='CPU input mount and bundle hash verification only; no GPU inference, runtime install or gameplay tested')
Path('/kaggle/working/input-probe.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result),flush=True)
